In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split


In [ ]:
data = pd.read_csv('../input/hospital-readmissions/train.csv')

data.head()


In [ ]:
y = data.readmitted

X = data.drop(columns=["readmitted"])


In [ ]:
train_X, val_X, train_y, val_y = train_test_split(
    X, y, random_state=1
)


In [ ]:
my_model = RandomForestClassifier(
    n_estimators=30,
    random_state=1
)

my_model.fit(train_X, train_y)


In [ ]:
# TreeExplainer is optimized for RandomForest
explainer = shap.TreeExplainer(my_model)


In [ ]:
def patient_risk_factors(patient_row, max_display=10):
    """
    patient_row: single-row DataFrame (same format as training data)
    max_display: number of most important features to show
    """

    # Compute SHAP values
    shap_values = explainer.shap_values(patient_row)

    # Use class 1 = readmitted
    shap_value_for_patient = shap_values[1]

    # Create waterfall explanation
    shap.plots.waterfall(
        shap.Explanation(
            values=shap_value_for_patient[0],
            base_values=explainer.expected_value[1],
            data=patient_row.iloc[0],
            feature_names=patient_row.columns
        ),
        max_display=max_display,
        show=True
    )


In [ ]:
# Select any single patient
patient = val_X.iloc[[0]]

patient_risk_factors(patient)
